# Monet Inference Multi-Test Notebook

这个 notebook 用于多轮推理测试：
- 仅在第一次运行时加载模型
- 后续可反复修改问题/图片并测试，不重复加载

In [ ]:
import os

# ========== Runtime Config ==========

MODEL_PATH = os.environ.get("MONET_MODEL_PATH", "NOVAglow646/Monet-7B")
LATENT_SIZE = int(os.environ.get("LATENT_SIZE", "10"))
TP = 1
GPU_MEMORY_UTILIZATION = 0.8

os.environ["LATENT_SIZE"] = str(LATENT_SIZE)

print(f"MODEL_PATH={MODEL_PATH}")
print(f"LATENT_SIZE={LATENT_SIZE}")

: 

In [ ]:
import re
import PIL.Image
from transformers import AutoProcessor

import inference.apply_vllm_monet
from inference.load_and_gen_vllm import (
    vllm_mllm_init,
    vllm_mllm_process_batch_from_messages,
    vllm_generate,
)

if "_MONET_RUNTIME" not in globals():
    print("Initializing Monet runtime (first load)...")
    mllm, sampling_params = vllm_mllm_init(
        MODEL_PATH,
        tp=TP,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    )
    processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
    _MONET_RUNTIME = {
        "mllm": mllm,
        "sampling_params": sampling_params,
        "processor": processor,
    }
else:
    print("Reusing cached Monet runtime.")

runtime = _MONET_RUNTIME

: 

In [ ]:
def _clean_latent_text(s: str) -> str:
    pattern = re.compile(r"(<abs_vis_token>)(.*?)(</abs_vis_token>)", flags=re.DOTALL)
    return pattern.sub(r"\1<latent>\3", s)

def run_case(question: str, image_path: str):
    image = PIL.Image.open(image_path).convert("RGB")
    conversations = [[{
        "role": "user",
        "content": [
            {"type": "text", "text": question},
            {"type": "image", "image": image},
        ]
    }]]

    inputs = vllm_mllm_process_batch_from_messages(conversations, runtime["processor"])
    outputs = vllm_generate(inputs, runtime["sampling_params"], runtime["mllm"])
    raw_text = outputs[0].outputs[0].text
    cleaned = _clean_latent_text(raw_text)

    print("===== RAW OUTPUT =====")
    print(raw_text)
    print("\n===== CLEANED OUTPUT =====")
    print(cleaned)
    return cleaned

In [ ]:
# Single test
question = "Question: Which car has the longest rental period? The choices are listed below:\n(A) DB11 COUPE.\n(B) V12 VANTAGES COUPES.\n(C) VANQUISH VOLANTE.\n(D) V12 VOLANTE.\n(E) The image does not feature the time. Put your final answer in \\boxed{}."
image_path = "images/example_question.png"

_ = run_case(question, image_path)

In [ ]:
# Batch test examples (edit freely)
test_cases = [
    {
        "question": "Question: Which car has the longest rental period? Put your final answer in \\boxed{}.",
        "image_path": "images/example_question.png",
    },
]

for i, case in enumerate(test_cases, 1):
    print(f"\n================ CASE {i} ================")
    _ = run_case(case["question"], case["image_path"])